# ML Benchmark — Household Electricity Consumption
**Dataset:** Africa Synth Energy (electrified households, Gold layer `gold.gold_ml_features`)  
**Task:** Regression — predict `monthly_electricity_kwh` from household characteristics  
**Models:** Baseline, Linear, Ridge, Random Forest, HistGradientBoosting, LightGBM, XGBoost, CatBoost  
**Method:** 5-fold cross-validation + held-out test set; Optuna tuning for the boosters; early stopping on an internal validation split (the test set is never used for fitting); SHAP on the best model.


In [ ]:
!pip install -q "duckdb==1.5.3" lightgbm xgboost catboost optuna shap scikit-learn pandas matplotlib python-dotenv

In [ ]:
import os
import warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap
import optuna
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
N_TRIALS = 20

MD_TOKEN = ""
try:
    from google.colab import userdata
    MD_TOKEN = userdata.get("MOTHERDUCK_TOKEN")
except Exception:
    pass
if not MD_TOKEN:
    MD_TOKEN = os.environ.get("MOTHERDUCK_TOKEN", "")
if not MD_TOKEN:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        MD_TOKEN = os.environ.get("MOTHERDUCK_TOKEN", "")
    except Exception:
        pass
assert MD_TOKEN, "MOTHERDUCK_TOKEN not found (Colab Secrets / env var / .env)"

## 1. Load data from the Gold layer


In [ ]:
con = duckdb.connect(f"md:energy_lakehouse?motherduck_token={MD_TOKEN}")
df = con.execute("SELECT * FROM gold.gold_ml_features").fetchdf()
print(f"Rows: {len(df):,} | Cols: {len(df.columns)}")
df.head(3)

In [ ]:
print(df["target_kwh"].describe().round(2))

## 2. Feature preparation


In [ ]:
TARGET = "target_kwh"
y = df[TARGET].astype(float)
X = df.drop(columns=["household_id", TARGET])

bool_cols = X.select_dtypes(include="bool").columns.tolist()
X[bool_cols] = X[bool_cols].astype(int)
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
for c in cat_cols:
    X[c] = X[c].astype("category")

print(f"Features: {X.shape[1]} | categorical: {cat_cols}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

## 3. Helper functions


In [ ]:
def metrics(yt, yp):
    return mean_absolute_error(yt, yp), root_mean_squared_error(yt, yp), r2_score(yt, yp)

def ohe_prep(scale):
    num = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
        ("num", num, num_cols)])

def cv_ohe(est):
    sc = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
    r = cross_validate(est, X_train, y_train, cv=kf, scoring=sc, n_jobs=-1)
    return {"cv_mae": -r["test_mae"].mean(), "cv_mae_std": r["test_mae"].std(),
            "cv_rmse": -r["test_rmse"].mean(), "cv_r2": r["test_r2"].mean()}

def make_native(kind, params, n):
    if kind == "lgbm":
        return lgb.LGBMRegressor(n_estimators=n, random_state=SEED, verbose=-1, **params)
    if kind == "xgb":
        return xgb.XGBRegressor(n_estimators=n, random_state=SEED, tree_method="hist",
                                enable_categorical=True, verbosity=0, **params)
    return CatBoostRegressor(iterations=n, random_seed=SEED, verbose=False, **params)

def fit_native(model, kind, Xt, yt, Xv, yv, early=True):
    if kind == "lgbm":
        model.fit(Xt, yt, eval_set=[(Xv, yv)] if early else None,
                  callbacks=[lgb.early_stopping(50, verbose=False)] if early else None)
    elif kind == "xgb":
        if early:
            model.set_params(early_stopping_rounds=50)
            model.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
        else:
            model.fit(Xt, yt, verbose=False)
    else:
        tp = Pool(Xt, yt, cat_features=cat_cols)
        if early:
            model.fit(tp, eval_set=Pool(Xv, yv, cat_features=cat_cols),
                      early_stopping_rounds=50, verbose=False)
        else:
            model.fit(tp, verbose=False)
    return model

def cv_native(kind, params):
    ma, rm, r2 = [], [], []
    for tr, va in kf.split(X_train):
        m = make_native(kind, params, 300)
        fit_native(m, kind, X_train.iloc[tr], y_train.iloc[tr], None, None, early=False)
        a, b, c = metrics(y_train.iloc[va], m.predict(X_train.iloc[va]))
        ma.append(a); rm.append(b); r2.append(c)
    return {"cv_mae": float(np.mean(ma)), "cv_mae_std": float(np.std(ma)),
            "cv_rmse": float(np.mean(rm)), "cv_r2": float(np.mean(r2))}

def tune(kind):
    k3 = KFold(n_splits=3, shuffle=True, random_state=SEED)
    def space(t):
        if kind == "lgbm":
            return {"learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "num_leaves": t.suggest_int("num_leaves", 15, 127),
                    "min_child_samples": t.suggest_int("min_child_samples", 10, 80),
                    "subsample": t.suggest_float("subsample", 0.6, 1.0),
                    "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0)}
        if kind == "xgb":
            return {"learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "max_depth": t.suggest_int("max_depth", 3, 10),
                    "min_child_weight": t.suggest_int("min_child_weight", 1, 10),
                    "subsample": t.suggest_float("subsample", 0.6, 1.0),
                    "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0)}
        return {"learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "depth": t.suggest_int("depth", 4, 10),
                "l2_leaf_reg": t.suggest_float("l2_leaf_reg", 1.0, 10.0)}
    def obj(t):
        p = space(t)
        ma = []
        for tr, va in k3.split(X_train):
            m = make_native(kind, p, 300)
            fit_native(m, kind, X_train.iloc[tr], y_train.iloc[tr], None, None, early=False)
            ma.append(mean_absolute_error(y_train.iloc[va], m.predict(X_train.iloc[va])))
        return float(np.mean(ma))
    st = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    st.optimize(obj, n_trials=N_TRIALS, timeout=180)
    print(f"  {kind}: best CV MAE={st.best_value:.3f}")
    return st.best_params

## 4. Baseline + one-hot models (Linear, Ridge, Random Forest, HistGB)


In [ ]:
rows = []
bp = np.full(len(y_test), y_train.mean())
a, b, c = metrics(y_test, bp)
rows.append({"model": "Baseline (mean)", "cv_mae": np.nan, "cv_mae_std": np.nan,
             "cv_rmse": np.nan, "cv_r2": np.nan, "test_mae": a, "test_rmse": b, "test_r2": c})

ohe = {"Linear Regression": (LinearRegression(), True),
       "Ridge": (Ridge(alpha=1.0, random_state=SEED), True),
       "Random Forest": (RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=SEED), False),
       "HistGradientBoosting": (HistGradientBoostingRegressor(random_state=SEED), False)}
fitted = {}
for name, (est, scale) in ohe.items():
    pipe = Pipeline([("prep", ohe_prep(scale)), ("model", est)])
    cvm = cv_ohe(pipe)
    pipe.fit(X_train, y_train)
    tm = metrics(y_test, pipe.predict(X_test))
    fitted[name] = ("ohe", pipe)
    rows.append({"model": name, **cvm, "test_mae": tm[0], "test_rmse": tm[1], "test_r2": tm[2]})
    print(f"{name:22s} CV MAE={cvm['cv_mae']:.2f} | Test MAE={tm[0]:.2f} R2={tm[2]:.4f}")

## 5. Gradient boosting with Optuna tuning (LightGBM, XGBoost, CatBoost)


In [ ]:
Xtr2, Xval2, ytr2, yval2 = train_test_split(X_train, y_train, test_size=0.15, random_state=SEED)
for name, kind in {"LightGBM (tuned)": "lgbm", "XGBoost (tuned)": "xgb", "CatBoost (tuned)": "catboost"}.items():
    best = tune(kind)
    cvm = cv_native(kind, best)
    m = make_native(kind, best, 2000)
    fit_native(m, kind, Xtr2, ytr2, Xval2, yval2, early=True)
    tm = metrics(y_test, m.predict(X_test))
    fitted[name] = (kind, m)
    rows.append({"model": name, **cvm, "test_mae": tm[0], "test_rmse": tm[1], "test_r2": tm[2]})
    print(f"{name:18s} CV MAE={cvm['cv_mae']:.2f} | Test MAE={tm[0]:.2f} R2={tm[2]:.4f}")

## 6. Comparison table & chart


In [ ]:
table = pd.DataFrame(rows)
ranked = table.dropna(subset=["cv_mae"]).sort_values("cv_mae")
best_name = ranked.iloc[0]["model"]
table["is_best"] = table["model"] == best_name
display(table.round(3))
print("Best model:", best_name)

d = table[table["model"] != "Baseline (mean)"].sort_values("test_mae")
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].barh(d["model"], d["test_mae"], color="steelblue"); ax[0].invert_yaxis()
ax[0].set_title("Test MAE (lower better)")
ax[1].barh(d["model"], d["test_r2"], color="seagreen"); ax[1].invert_yaxis()
ax[1].set_title("Test R2 (higher better)")
plt.tight_layout(); plt.savefig("model_comparison.png", dpi=150); plt.show()

## 7. Best model — Predicted vs Actual & SHAP


In [ ]:
best_kind, best_model = fitted[best_name]
pred = best_model.predict(X_test)
_, _, r2 = metrics(y_test, pred)

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, pred, alpha=0.3, s=10, color="steelblue")
lim = max(y_test.max(), pred.max())
ax.plot([0, lim], [0, lim], "r--", label="Perfect"); ax.legend()
ax.set_xlabel("Actual kWh"); ax.set_ylabel("Predicted kWh")
ax.set_title(f"Predicted vs Actual — {best_name} — R2={r2:.4f}")
plt.tight_layout(); plt.savefig("predicted_vs_actual.png", dpi=150); plt.show()

if best_kind in ("lgbm", "xgb", "catboost"):
    Xs = X_test.sample(n=min(500, len(X_test)), random_state=SEED)
    expl = shap.TreeExplainer(best_model)
    sv = expl.shap_values(Pool(Xs, cat_features=cat_cols)) if best_kind == "catboost" else expl.shap_values(Xs)
    plt.figure(figsize=(9, 6)); shap.summary_plot(sv, Xs, show=False)
    plt.tight_layout(); plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight"); plt.show()
    joblib.dump(best_model, "best_model.pkl")

## 8. Export predictions & metrics to MotherDuck


In [ ]:
results = X_test.copy()
results["actual_kwh"] = y_test.values
results["predicted_kwh"] = np.round(pred, 2)
results["error_kwh"] = np.round(pred - y_test.values, 2)
results["abs_error_kwh"] = np.round(np.abs(pred - y_test.values), 2)
for c in results.select_dtypes(include="category").columns:
    results[c] = results[c].astype(str)
con.register("_pred_staging", results)
con.execute("CREATE SCHEMA IF NOT EXISTS gold;")
con.execute("CREATE OR REPLACE TABLE gold.gold_predictions AS SELECT * FROM _pred_staging;")

mt = table[["model", "cv_mae", "cv_mae_std", "cv_rmse", "cv_r2", "test_mae", "test_rmse", "test_r2", "is_best"]]
con.register("_metrics_staging", mt)
con.execute("CREATE OR REPLACE TABLE gold.gold_model_metrics AS SELECT * FROM _metrics_staging;")
print("Exported gold.gold_predictions and gold.gold_model_metrics. Best:", best_name)
con.close()